# Packages

In [ ]:
%pip install sentence-transformers deepeval==3.9.1 accelerate==1.13.0 transformers==5.3.0 langchain-mistralai

# Imports

In [ ]:
import torch
from deepeval import evaluate
from deepeval.evaluate import AsyncConfig
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.models.base_model import DeepEvalBaseLLM
import asyncio
import csv
import os
import json
from tqdm import tqdm
from google.colab import userdata
from google.colab import drive

drive.mount('/content/drive')

# Load Test Cases

In [ ]:
base_dir = "/content/drive/MyDrive/RAG Outputs GPT"

test_cases = {}

for filename in os.listdir(base_dir):
  if filename[:4] == 'rag6' and filename[-4:] == '.csv':
    test_cases[filename] = []

    with open(f"{base_dir}/{filename}", "r") as csv_file:
      reader = csv.DictReader(csv_file)

      for row in reader:
        if "input" in row and "actual_output" in row and "retrieval_context" in row:
            test_cases[filename].append(
                LLMTestCase(
                    input=row["input"].strip(),
                    actual_output=row["actual_output"].strip(),
                    retrieval_context=row["retrieval_context"].split("\n\n---\n\n")
                )
            )

      print(f"Loaded test cases from {filename}")

# Evaluate

### HuggingFace

In [ ]:
# from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer

# class EvalModel(DeepEvalBaseLLM):
#     def __init__(self, model, tokenizer):
#         self.pipe = None
#         self.load_model(model, tokenizer)

#     def load_model(self, model, tokenizer):
#         self.pipe = pipeline(
#             "text-generation",
#             model=model,
#             tokenizer=tokenizer,
#         )
#         return self.pipe

#     def _generate(self, prompt: str) -> str:
#         messages = [{"role": "user", "content": prompt}]

#         outputs = self.pipe(
#             messages,
#             max_new_tokens=512,
#             temperature=0.1,
#             do_sample=True,
#             pad_token_id=self.pipe.tokenizer.eos_token_id
#         )

#         return outputs[0]["generated_text"][-1]["content"].strip()

#     def generate(self, prompt: str, **kwargs) -> str:
#         return self._generate(prompt)

#     async def a_generate(self, prompt: str, **kwargs) -> str:
#         loop = asyncio.get_running_loop()
#         return await loop.run_in_executor(None, self._generate, prompt)

#     def generate_raw_response(self, prompt: str, **kwargs) -> str:
#         response = self._generate(prompt)
#         return (response, 0.0)

#     async def a_generate_raw_response(self, prompt: str, **kwargs) -> str:
#         loop = asyncio.get_running_loop()
#         response = await loop.run_in_executor(None, self._generate, prompt)
#         return (response, 0.0)

#     def get_model_name(self) -> str:
#         return "Qwen3.5"

# model = AutoModelForCausalLM.from_pretrained(
#     "Qwen/Qwen3.5-4B",
#     device_map="auto",
#     torch_dtype=torch.float16,
#     trust_remote_code=False
# )

# tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3.5-4B")

# evaluator = EvalModel(model, tokenizer)

### Mistral

In [ ]:
from langchain_mistralai import ChatMistralAI

class EvalModel(DeepEvalBaseLLM):
    def __init__(self, model):
        self.model = model

    def load_model(self):
        return self.model

    def _generate(self, prompt: str) -> str:
        output = self.load_model().invoke(prompt).content

        return output

    def generate(self, prompt: str, **kwargs) -> str:
        return self._generate(prompt)

    async def a_generate(self, prompt: str, **kwargs) -> str:
        loop = asyncio.get_running_loop()
        return await loop.run_in_executor(None, self._generate, prompt)

    def generate_raw_response(self, prompt: str, **kwargs) -> str:
        response = self._generate(prompt)
        return (response, 0.0)

    async def a_generate_raw_response(self, prompt: str, **kwargs) -> str:
        loop = asyncio.get_running_loop()
        response = await loop.run_in_executor(None, self._generate, prompt)
        return (response, 0.0)

    def get_model_name(self) -> str:
        return "Mistral"

mistral = ChatMistralAI(
    model="mistral-small-2506",
    temperature=0,
    max_retries=2,
    api_key=userdata.get('MISTRAL_API_KEY')
)

evaluator = EvalModel(mistral)

### Gemini

In [ ]:
# from deepeval.models import GeminiModel
# from google.colab import userdata

# evaluator = GeminiModel(
#     model="gemini-3.1-flash-lite-preview",
#     # model="gemini-2.5-flash-lite",
#     # model="gemma-3-27b-it",
#     # model="gemini-2.5-pro",
#     # model="gemini-3.1-flash-lite-preview",
#     # model="gemini-2.0-flash",
#     api_key=userdata.get('GEMINI_API_KEY'),
#     temperature=0
# )

### OpenAI

In [ ]:
# from deepeval.models import GPTModel
# from google.colab import userdata

# evaluator = GPTModel(
#     api_key=userdata.get('OPENAI_API_KEY'),
#     model="gpt-4.1",
#     temperature=0
# )

## DeepEval

In [ ]:
# Increase per-task timeout (in seconds)
os.environ["DEEPEVAL_PER_TASK_TIMEOUT_SECONDS_OVERRIDE"] = "300"

# Increase buffer time for gathering results
os.environ["DEEPEVAL_TASK_GATHER_BUFFER_SECONDS_OVERRIDE"] = "60"

def evaluate_test_cases(testcases):
  factuality = GEval(
      name="Factuality & Grounding",
      criteria="Determine if actual_output is factually correct and fully grounded in retrieval_context.",
      evaluation_steps=[
          "List all factual claims made in the actual_output.",
          "Verify each claim is directly supported by retrieval_context.",
          "Penalize hallucinations, speculation, or unsupported details.",
          "Score 1-10: 10=perfectly factual/grounded, 1=fully hallucinated."
      ],
      evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT, LLMTestCaseParams.RETRIEVAL_CONTEXT],
      model=evaluator,
      threshold=0.7
  )

  recall = GEval(
      name="Recall",
      criteria="Assess if retrieval_context contains all information needed to fully answer the input query.",
      evaluation_steps=[
          "Identify key facts required to answer the input query.",
          "Check if retrieval_context includes these key facts.",
          "Penalize missing critical information needed for complete answer.",
          "Score 1-10: 10=context fully supports ideal answer, 1=critical info missing."
      ],
      evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.RETRIEVAL_CONTEXT],
      model=evaluator,
      threshold=0.7
  )

  f1_score = GEval(
      name="F1 Score",
      criteria="Evaluate the balance between precision (no extra info) and recall (all necessary info) in the actual_output relative to the provided context.",
      evaluation_steps=[
          "Identify necessary components of an answer based on the context.",
          "Check if actual_output contains unnecessary or incorrect information (Precision).",
          "Check if actual_output contains all necessary information found in the context (Recall).",
          "Calculate a score representing the balance (F1) of these two factors."
      ],
      evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT, LLMTestCaseParams.RETRIEVAL_CONTEXT],
      model=evaluator,
      threshold=0.7
  )

  return evaluate(
      test_cases=testcases,
      async_config=AsyncConfig(run_async=True, throttle_value=3, max_concurrent=3),
      metrics=[f1_score, recall, factuality]
  ).model_dump()

for filename, file_test_cases in tqdm(iterable=test_cases.items(), total=len(test_cases.keys())):
  results = evaluate_test_cases(file_test_cases)

  json.dump(results, open(f"{base_dir}/{filename}.json", "w"))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Initialize data storage
file_names = []
factuality_avgs = []
recall_avgs = []
f1_avgs = []

for filename in sorted(os.listdir(base_dir)):
    if filename.endswith('.json'):
        with open(os.path.join(base_dir, filename), 'r') as f:
            data = json.load(f)

            # Extract scores from test_results
            # Based on the previous cell, metrics are: [f1, recall, factuality]
            f1_scores = [r['metrics_data'][0]['score'] for r in data['test_results']]
            recall_scores = [r['metrics_data'][1]['score'] for r in data['test_results']]
            fact_scores = [r['metrics_data'][2]['score'] for r in data['test_results']]

            file_names.append(filename.replace('.json', ''))
            f1_avgs.append(np.mean(f1_scores))
            recall_avgs.append(np.mean(recall_scores))
            factuality_avgs.append(np.mean(fact_scores))

# Plotting
x = np.arange(len(file_names))
width = 0.25

fig, ax = plt.subplots(figsize=(12, 6))
rects1 = ax.bar(x - width, factuality_avgs, width, label='Faithfulness (Factuality)')
rects2 = ax.bar(x, f1_avgs, width, label='F1 Score')
rects3 = ax.bar(x + width, recall_avgs, width, label='Recall')

ax.set_ylabel('Average Score')
ax.set_title('RAG Evaluation Metrics by File')
ax.set_xticks(x)
ax.set_xticklabels(file_names, rotation=45, ha='right')
ax.legend()
ax.set_ylim(0, 1)  # GEval typically scales 1-10

plt.tight_layout()
plt.show()

# Results

In [ ]:
import os
import json

base_dir = "/content/drive/MyDrive/RAG Outputs GPT"

eval_results = {}

for filename in os.listdir(base_dir):
  if filename[-5:] == ".json":
    eval_results[filename] = json.load(open(f"{base_dir}/{filename}", "r"))

In [ ]:
import re
import pandas as pd

pattern = r'rag\d_eval\.csv'

rows = []

for filename, data in eval_results.items():
  if re.match(pattern, filename):
    variant_name = filename.replace('.csv.json', '').replace('.json', '')

    print(f"Processing {variant_name}")

    for result in data:
        metrics = result.get('metrics_data', [])

        f1_score = None
        recall_score = None
        factuality_score = None
        correctness_score = None

        # Use partial matching because names like 'F1 Score [GEval]' are used
        for metric in metrics:
            name = metric.get('name', '')
            score = metric.get('score')

            if name.startswith('F1 Score'):
                f1_score = score
            elif name.startswith('Recall'):
                recall_score = score
            elif name.startswith('Factuality & Grounding'):
                factuality_score = score
            elif name.startswith('Correctness'):
                correctness_score = score

        rows.append({
            'RAG Variant': variant_name,
            'F1 Score': f1_score,
            'Recall': recall_score,
            'Factuality & Grounding': factuality_score,
            'Correctness': correctness_score
        })

df_scores = pd.DataFrame(rows)

# Calculate descriptive statistics for each RAG variant
summary_stats = df_scores.groupby('RAG Variant').describe().transpose()

# Alternatively, for a cleaner overview of specific metrics:
variant_comparison = df_scores.groupby('RAG Variant').agg(['mean', 'median', 'std', 'min', 'max'])

print("Descriptive Statistics for RAG Variants:")
display(variant_comparison)

# Print a summary table specifically for the means to easily compare performance
print("\nMean Scores Comparison:")
means_df = df_scores.groupby('RAG Variant').mean().sort_values(by='Factuality & Grounding', ascending=False)

means_df = means_df.sort_index()

display(means_df)

In [ ]:
import matplotlib.pyplot as plt

# Plotting the means for each RAG variant
ax = means_df.plot(kind='bar', figsize=(14, 7))

# Adding values on top of each bar
for container in ax.containers:
    ax.bar_label(container, fmt='%.2f', padding=3, rotation=45, fontsize=8)

plt.title('Mean Evaluation Scores by RAG Variant')
plt.ylabel('Average Score (0-1)')
plt.xlabel('RAG Variant')
plt.xticks(rotation=45)
plt.legend(title='Metrics', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.ylim(0, 1.1)  # Adjust limit to make room for labels
plt.tight_layout()

plt.savefig('means.png')
plt.show()

In [ ]:
means_df.to_markdown()

# Semantic Similarity b/t `input` and `retrieval_context`

In [ ]:
import pandas as pd
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer('all-MiniLM-L6-v2')

In [ ]:
import numpy as np
import os
import pandas as pd

pattern = r'rag\d_eval\.csv'

base_dir = '/content/drive/MyDrive/RAG Outputs GPT'

for filename in os.listdir(base_dir):
  if re.match(pattern, filename):
    file_path = f'{base_dir}/{filename}'

    df_rag = pd.read_csv(file_path)

    # Prepare texts (ensure they are strings and handle potential NaNs)
    inputs = df_rag['input'].astype(str).tolist()
    contexts = df_rag['retrieval_context'].astype(str).tolist()

    # Compute embeddings
    input_embeddings = model.encode(inputs, convert_to_tensor=True)
    context_embeddings = model.encode(contexts, convert_to_tensor=True)

    # Compute cosine similarities
    # util.cos_sim returns a matrix; we take the diagonal for pair-wise comparison
    cosine_scores = util.cos_sim(input_embeddings, context_embeddings)
    similarities = cosine_scores.diag().tolist()

    # Update DataFrame
    df_rag['semantic_similarity'] = similarities

    filtered_df = df_rag[df_rag['type'].isin(['factoid', 'list'])][['id', 'semantic_similarity']]

    mean_similarity = np.mean(filtered_df["semantic_similarity"].values)

    print(f"Mean similarity for {filename}: {mean_similarity}")

In [ ]:
from pprint import pprint

mean_similarities = {
    "rag1": 0.45454594665030135,
    "rag2": 0.3989769744486371,
    "rag3": 0.46002109240316136,
    "rag4": 0.46002109240316136,
    "rag5": 0.46002109240316136,
    "rag6": 0.46002109240316136
}

pprint(mean_similarities)